# Wasted Wind - 2025 UK Wind Farm Curtailment

## Project Overview

This project explores publicly available data that outlines transactions between the National Energy System Operator (NESO) and UK-based windfarm operators, specifically reductions in wind generation that occur as a result of system constraints (wind curtailments).

### Key Questions

- Where do wind curtailments generally occur in the UK?
- How much energy is curtailed and how much does it cost to curtail this energy?

### Data Sources

- Elexon Insights BMRS API for bidding and pricing data and for basic BMU metadata
- Power Station Dictionary for BMU location data

### Methods

- Requested all relevant data for 2025 and performed SQL queries to process and combine data to calculate total expected and annual energy usage, and the cost of this to the system operator.
- Combined with location data from Power Station Dictionary
- Visualised data using Flourish

### Key Findings

- £375 million was spent paying windfarms to reduce their wind generation from their expected output
- A total of 10.9 TWh was curtailed
- Almost all curtailment occurs in Scotland north of the B6 boundary, highlighting the bottleneck that this poses

### 1. Data Acquisition

The most important data to answer the question of wind curtailment is **"Bid-Offer-Accepted-Level-Flagged" (BOALF) data**, which contains accepted bid-offer data in instances where revised bids are given by NESO when they realise they cannot meet their previous agreements, for example due to system constraints. 

This data is according to the respective Balancing Mechanism Unit (BMU), for which metadata exists in the **BMUnits** dataset.

The expected output of a given BMU is provided as the Final Physical Notification (FPN), which can be obtained via the **Physical Notification (PN)** dataset. 

The price per MW of any change in output relative to the FPN is given in the **"Bid-Order-Data" (BOD)** dataset.

Finally, the location data of many (but not all) BMUs can be obtained from the dictionary files available in **Power Station Dictionary**

Some of the API datasets allow for collecting all data for a given year at once, while some require collecting them monthly.

In [1]:
import requests
import pandas as pd
import csv
import duckdb
import datetime
from pathlib import Path
from dateutil.relativedelta import relativedelta
import itertools
import json


start_date = '2025-01-01T00:00Z'
end_date = '2026-01-01T00:00Z'

params = {
    'from': start_date,
    'to': end_date
}

# Fetch Bid-Offer-Acceptance-Flagged (BOALF) data

if not Path('data/bid_data.json').exists():
    print("Fetching BOALF data...")
    response = requests.get('https://data.elexon.co.uk/bmrs/api/v1/datasets/BOALF/stream', params=params)
    with open('data/bid_data.json', 'w') as f:
        f.write(response.text)
else:
    print("BOALF data already exists, skipping")

# Fetch Balancing Mechanism Unit (BMU) data

if not Path('data/bmu_data.json').exists():
    print("Fetching BMU data...")
    response = requests.get('https://data.elexon.co.uk/bmrs/api/v1/reference/bmunits/all')
    with open('data/bmu_data.json', 'w') as f:
        f.write(response.text)
else:
    print("BMU data already exists, skipping")

# Fetch Physical Notification (PN) data

if not Path('data/pn_data.json').exists():
    print('Fetching PN data...')

    start = datetime.date(2025,1,1)
    end = datetime.date(2026,1,1)
    all_pn = []

    while start < end:
        start_date = start
        end_date = start + relativedelta(months=1)
        params = {
        'from': start_date.strftime('%Y-%m-%dT%H:%MZ'),
        'to': end_date.strftime('%Y-%m-%dT%H:%MZ')
        }
        response = requests.get(
        'https://data.elexon.co.uk/bmrs/api/v1/datasets/PN/stream',
        params = params
        )
        print(f"Fetched {start.strftime('%Y-%m')}: {response.status_code}")
        all_pn.append(response.json())
        start = end_date

    flat = list(itertools.chain.from_iterable(all_pn))
    with open('data/pn_data.json', 'w') as f:
        json.dump(flat, f)

else:
    print('PN data found, skipping')

# Fetch Bid-Offer price (BOD) data

if not Path('data/price_data.json').exists():
    print('Fetching Price data...')

    start = datetime.date(2025,1,1)
    end = datetime.date(2026,1,1)

    all_prices = []

    while start < end:
        start_date = start
        end_date = start + relativedelta(months=1)
        params = {
        'from': start_date.strftime('%Y-%m-%dT%H:%MZ'),
        'to': end_date.strftime('%Y-%m-%dT%H:%MZ')
        }
        response = requests.get(
        'https://data.elexon.co.uk/bmrs/api/v1/datasets/BOD/stream',
        params = params
        )
        print(f"Fetched {start.strftime('%Y-%m')}: {response.status_code}")
        all_prices.append(response.json())
        start = end_date

    flat = list(itertools.chain.from_iterable(all_prices))
    with open('data/price_data.json', 'w') as f:
        json.dump(flat, f)
else:
    print('Price data found, skipping')

BOALF data already exists, skipping
BMU data already exists, skipping
PN data found, skipping
Price data found, skipping


### 2. Data Exploration
We can use DuckDB moving forward to work easily with the large datasets.

First we can look through each dataset to get a better understanding of them, beginning with the fuel types in the BMU data:

In [2]:
con = duckdb.connect()

con.sql("""
        SELECT fuelType, COUNT(*) FROM 'data/bmu_data.json' GROUP BY fuelType
        """).show()

┌──────────┬──────────────┐
│ fuelType │ count_star() │
│ varchar  │    int64     │
├──────────┼──────────────┤
│ COAL     │           10 │
│ CCGT     │           65 │
│ NPSHYD   │           36 │
│ INTVKL   │            2 │
│ INTNED   │            1 │
│ INTFR    │            1 │
│ INTIFA2  │            1 │
│ WIND     │          257 │
│ INTNEM   │            1 │
│ OTHER    │           66 │
│ PS       │           16 │
│ INTNSL   │            1 │
│ INTIRL   │            1 │
│ INTELEC  │            2 │
│ NULL     │         2374 │
│ BIOMASS  │           18 │
│ INTGRNL  │            2 │
│ INTEW    │            1 │
│ OCGT     │           23 │
│ NUCLEAR  │           16 │
├──────────┴──────────────┤
│ 20 rows       2 columns │
└─────────────────────────┘



It's possible that the NULL category contains more windfarms, but these will have to be ignored for now.

Next, we can look at the BOALF data:

In [3]:
con.sql("""
        SELECT * FROM 'data/bid_data.json' LIMIT 10
        """).show()

┌─────────┬────────────────┬──────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬───────────┬─────────┬──────────────────┬─────────────────────┬──────────────┬─────────┬───────────────┬──────────┬─────────┬────────────────────┬───────────┐
│ dataset │ settlementDate │ settlementPeriodFrom │ settlementPeriodTo │      timeFrom       │       timeTo        │ levelFrom │ levelTo │ acceptanceNumber │   acceptanceTime    │ deemedBoFlag │ soFlag  │ amendmentFlag │ storFlag │ rrFlag  │ nationalGridBmUnit │  bmUnit   │
│ varchar │      date      │        int64         │       int64        │      timestamp      │      timestamp      │   int64   │  int64  │      int64       │      timestamp      │   boolean    │ boolean │    varchar    │ boolean  │ boolean │      varchar       │  varchar  │
├─────────┼────────────────┼──────────────────────┼────────────────────┼─────────────────────┼─────────────────────┼───────────┼─────────┼──────────────────┼──────────────────

While there are "Settlement Periods" which are 30-minute increments, the BOALF data spans any number of minutes and across settlements periods, so this will need to be handled carefully. Also, the system operator often can accept multiple offers for a given time period, updating as time goes on. This will mean multiple acceptance numbers for a given time period, where previous acceptance numbers need to be filtered out.

Next, the BOD pricing data:

In [4]:
con.sql("""
        SELECT * FROM 'data/price_data.json' LIMIT 10
        """).show()

┌─────────┬────────────────┬──────────────────┬─────────────────────┬───────────┬─────────────────────┬─────────┬────────┬────────┬────────┬────────────────────┬────────────┐
│ dataset │ settlementDate │ settlementPeriod │      timeFrom       │ levelFrom │       timeTo        │ levelTo │ pairId │ offer  │  bid   │ nationalGridBmUnit │   bmUnit   │
│ varchar │      date      │      int64       │      timestamp      │   int64   │      timestamp      │  int64  │ int64  │ double │ double │      varchar       │  varchar   │
├─────────┼────────────────┼──────────────────┼─────────────────────┼───────────┼─────────────────────┼─────────┼────────┼────────┼────────┼────────────────────┼────────────┤
│ BOD     │ 2025-02-01     │                1 │ 2025-02-01 00:00:00 │       -16 │ 2025-02-01 00:30:00 │     -16 │     -1 │  179.0 │   85.0 │ ABERU-1            │ E_ABERDARE │
│ BOD     │ 2025-02-01     │                1 │ 2025-02-01 00:00:00 │        16 │ 2025-02-01 00:30:00 │      16 │      1 │  1

This data follows settlements periods, but the pairId, levelFrom, and levelTo require more attention. Each BM Unit provides up to 5 bands above the Operating Baseline (represented by PN) and 5 below, with bids, offers, and MW level thresholds for each of these. Since we are only looking at curtailments, we can filter for pairId < 0. When we come to calculating the actual costs, we will need to be mindful of the final cost spanning multiple bands.

### 3. Data Analysis

The first steps are to:

- Filter for soFlag = True (flagged as a system constraint) and fuelType = 'WIND'
- Filter for the most recent acceptance a given time period in the BOALF data
- Create a transaction table that joins on the Physical Notification (PN) data to each BOALF row, giving the expected vs. actual power generation

For PN data, an ASOF join is used as the time periods will not exactly line up. The PN data typically does not fluctuate significantly in short time periods, so this is a reasonable approximation

In [5]:
con.sql("""
    CREATE OR REPLACE VIEW latest_acceptances AS (
        SELECT *
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (
                    PARTITION BY nationalGridBmUnit, timeFrom
                    ORDER BY acceptanceNumber DESC
                ) AS rn
            FROM 'data/bid_data.json'
        )
        WHERE rn = 1
    );

    CREATE OR REPLACE VIEW transactions AS (
        SELECT
            DATE_TRUNC('day',bids.timeFrom) as settlementDate,
            bids.settlementPeriodFrom AS boalf_setFrom,
            bids.settlementPeriodTo AS boalf_setTo,
            pn.settlementPeriod AS pn_setPeriod,
            bids.timeFrom AS boalf_timeFrom,
            bids.timeTo AS boalf_timeTo,
            bids.levelFrom AS boalf_levelFrom,
            bids.levelTo AS boalf_levelTo,
            acceptanceNumber,
            bids.nationalGridBmUnit,
            fuel.fuelType,
            fuel.leadPartyName,
            gspGroupId,
            pn.timeFrom AS pn_timeFrom,
            pn.timeTo AS pn_timeTo,
            pn.levelFrom AS pn_levelFrom,
            pn.levelTo AS pn_levelTo,
            date_part('epoch', bids.timeTo - bids.timeFrom) / 60.0 AS elapsed_mins
        FROM latest_acceptances AS bids
        LEFT JOIN 'data/bmu_data.json' AS fuel
            ON bids.nationalGridBmUnit = fuel.nationalGridBmUnit
        ASOF JOIN 'data/pn_data.json' AS pn
            ON pn.nationalGridBmUnit = bids.nationalGridBmUnit
            AND pn.timeFrom <= bids.timeFrom
        WHERE fuel.fuelType = 'WIND' 
            AND bids.soFlag = TRUE
        ORDER BY boalf_timeFrom
    );
        
        SELECT * FROM transactions LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬───────────────┬─────────────┬──────────────┬─────────────────────┬─────────────────────┬─────────────────┬───────────────┬──────────────────┬────────────────────┬──────────┬──────────────────────────────┬────────────┬─────────────────────┬─────────────────────┬──────────────┬────────────┬──────────────┐
│ settlementDate │ boalf_setFrom │ boalf_setTo │ pn_setPeriod │   boalf_timeFrom    │    boalf_timeTo     │ boalf_levelFrom │ boalf_levelTo │ acceptanceNumber │ nationalGridBmUnit │ fuelType │        leadPartyName         │ gspGroupId │     pn_timeFrom     │      pn_timeTo      │ pn_levelFrom │ pn_levelTo │ elapsed_mins │
│      date      │     int64     │    int64    │    int64     │      timestamp      │      timestamp      │      int64      │     int64     │      int64       │      varchar       │ varchar  │           varchar            │  varchar   │      timestamp      │      timestamp      │    int64     │   int64    │    double    │
├────────────────┼──────────

With the transactions table created, we can calculate energy from the power data using the elapsed_mins, both according to the BOALF and PN data, with the difference between them being the curtailed energy amount:

In [ ]:
con.sql("""
        CREATE OR REPLACE VIEW curtailment_table AS (
        SELECT
                nationalGridBmUnit,
                gspGroupId,
                settlementDate,
                boalf_timeFrom,
                boalf_timeTo,
                boalf_setFrom,
                boalf_setTo,
                (boalf_levelFrom + boalf_levelTo) / 2 as boalf_level,
                PN_levelFrom as pn_level,
                leadPartyName,
                elapsed_mins,
                boalf_levelFrom * DATE_PART('epoch',BOALF_timeTo - BOALF_timeFrom)/3600.0 as BOALF_mwh,
                pn_levelFrom * DATE_PART('epoch',BOALF_timeTo - BOALF_timeFrom) / 3600.0 as PN_mwh,
                pn_levelFrom * DATE_PART('epoch',BOALF_timeTo - BOALF_timeFrom) / 3600.0 - boalf_levelFrom * DATE_PART('epoch',BOALF_timeTo - BOALF_timeFrom)/3600.0 as curtailment_mwh
        FROM transactions
        WHERE (BOALF_levelFrom < PN_levelFrom) OR (BOALF_levelFrom < PN_levelTo) OR (BOALF_levelTo < PN_levelFrom) OR (BOALF_levelTo < PN_levelFrom)
        ORDER BY BOALF_timeFrom
        );

        SELECT * FROM curtailment_table
        """).show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬────────────┬────────────────┬─────────────────────┬─────────────────────┬───────────────┬─────────────┬─────────────┬──────────┬──────────────────────────────┬──────────────┬────────────────────┬────────────────────┬─────────────────────┐
│ nationalGridBmUnit │ gspGroupId │ settlementDate │   boalf_timeFrom    │    boalf_timeTo     │ boalf_setFrom │ boalf_setTo │ boalf_level │ pn_level │        leadPartyName         │ elapsed_mins │     BOALF_mwh      │       PN_mwh       │   curtailment_mwh   │
│      varchar       │  varchar   │      date      │      timestamp      │      timestamp      │     int64     │    int64    │   double    │  int64   │           varchar            │    double    │       double       │       double       │       double        │
├────────────────────┼────────────┼────────────────┼─────────────────────┼─────────────────────┼───────────────┼─────────────┼─────────────┼──────────┼──────────────────────────────┼──────────────┼─────────────────

The BOALF and PN power levels can be used to calculate the curtailment cost using the price bands given in the BOD data, ensuring that the amount charged is according to the amount of energy generated within that band alone:

In [ ]:
con.sql("""
        CREATE OR REPLACE VIEW ct_priced AS (
        SELECT
                *,
                -1 * (CASE   WHEN (level_difference - previous_bands_total) > 0 AND (level_difference - previous_bands_total - price_band) > 0 THEN price_band * bid * elapsed_mins/60
                        WHEN (level_difference - previous_bands_total) > 0 AND (level_difference - previous_bands_total - price_band) < 0 THEN (level_difference - previous_bands_total) * bid * elapsed_mins/60
                        ELSE 0 END) as curtailed_cost
        FROM
        (SELECT
                leadPartyName,
                ct.nationalGridBmUnit,
                ct.settlementDate,
                boalf_timeFrom,
                boalf_setFrom,
                boalf_level,
                pn_level,
                pn_level - boalf_level as level_difference,
                elapsed_mins,
                boalf_mwh,
                pn_mwh,
                curtailment_mwh,
                pairId,
                (cprice.levelFrom * -1) as price_band,
                (cprice.levelTo * -1) as price_band_to,
                COALESCE((SUM(price_band) OVER (
                        PARTITION BY cprice.nationalGridBmUnit, cprice.settlementDate, ct.boalf_timeFrom
                        ORDER BY pairId DESC
                        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                )),0) AS previous_bands_total,
                offer,
                bid
        FROM curtailment_table as ct
        LEFT JOIN (SELECT * FROM 'data/price_data.json' WHERE pairId < 0) as cprice
        ON ct.nationalGridBmUnit = cprice.nationalGridBmUnit AND cprice.settlementDate = ct.settlementDate AND  cprice.settlementPeriod = ct.boalf_setFrom
        ));

        SELECT * FROM ct_priced WHERE nationalGridBmUnit = 'BHLAW-1'
        
        """).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬────────────────────┬────────────────┬─────────────────────┬───────────────┬─────────────┬──────────┬──────────────────┬──────────────┬────────────────────┬────────────────────┬────────────────────┬────────┬────────────┬───────────────┬──────────────────────┬────────┬────────┬────────────────────┐
│   leadPartyName    │ nationalGridBmUnit │ settlementDate │   boalf_timeFrom    │ boalf_setFrom │ boalf_level │ pn_level │ level_difference │ elapsed_mins │     BOALF_mwh      │       PN_mwh       │  curtailment_mwh   │ pairId │ price_band │ price_band_to │ previous_bands_total │ offer  │  bid   │   curtailed_cost   │
│      varchar       │      varchar       │      date      │      timestamp      │     int64     │   double    │  int64   │      double      │    double    │       double       │       double       │       double       │ int64  │   int64    │     int64     │        int128        │ double │ double │       double       │
├────────────────────┼───────────────

Ultimately, we will need to bin the readings to a consistent time grid to be able to track both the expected generation (PN data, according to settlementPeriod) and actual generation (arbitrary times). Since we are looking across a year, we can simply summarise daily:

In [8]:
con.sql("""
        CREATE OR REPLACE VIEW daily_curtailment AS (
            SELECT
                nationalGridBmUnit,
                DATE_TRUNC('day', boalf_timeFrom) AS day,
                SUM(curtailment_mwh) as curtailed_mwh,
                SUM(curtailed_cost) as curtailed_cost,
                SUM(curtailed_cost)/SUM(curtailment_mwh) as cost_per_mwh
            FROM ct_priced
            GROUP BY nationalGridBmUnit, DATE_TRUNC('day', boalf_timeFrom)
        );

        SELECT * FROM daily_curtailment
        """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬────────────┬────────────────────┬────────────────────┬────────────────────┐
│ nationalGridBmUnit │    day     │   curtailed_mwh    │   curtailed_cost   │    cost_per_mwh    │
│      varchar       │    date    │       double       │       double       │       double       │
├────────────────────┼────────────┼────────────────────┼────────────────────┼────────────────────┤
│ AKGLW-3            │ 2025-01-13 │              818.9 │             5715.4 │  6.979362559531078 │
│ BABAW-1            │ 2025-09-21 │  624.3999999999999 │ 50060.316666666666 │  80.17347320093958 │
│ BRYBW-1            │ 2025-03-05 │  707.5833333333333 │           45368.75 │   64.1178895300907 │
│ BRYBW-1            │ 2025-05-04 │ 127.81666666666666 │             9430.0 │  73.77754596427175 │
│ CAIRW-2            │ 2025-12-10 │  580.4333333333334 │  48778.02416666667 │   84.0372552116235 │
│ CGTHW-1            │ 2025-03-29 │             1566.0 │         63311.1675 │  40.42858716475096 │
│ CRMLW-1 

Now we can do the same with the PN data and combine the two:

In [9]:
con.sql("""
        
        CREATE OR REPLACE VIEW daily_pn AS (
            SELECT
                pn.nationalGridBmUnit,
                DATE_TRUNC('day', timeFrom) AS day,
                SUM((levelFrom + levelTo) / 2.0 * date_part('epoch', timeTo - timeFrom) / 3600.0) AS pn_mwh
            FROM 'data/pn_data.json' as pn
            LEFT JOIN 'data/bmu_data.json' as fuel
            ON pn.nationalGridBmUnit = fuel.nationalGridBmUnit
            WHERE fuel.fuelType = 'WIND'
            GROUP BY pn.nationalGridBmUnit, DATE_TRUNC('day', timeFrom)
        );
        
        

        CREATE OR REPLACE VIEW daily AS 
        (SELECT
            pn.day,
            pn.nationalGridBmUnit,
            SUM(pn.pn_mwh) as expected_mwh,
            SUM(pn.pn_mwh) - COALESCE(SUM(c.curtailed_mwh),0) as actual_mwh,
            COALESCE(SUM(c.curtailed_mwh),0) as curtailed_mwh,
            COALESCE(SUM(curtailed_cost),0) as curtailed_cost
        FROM daily_pn as pn
        LEFT JOIN daily_curtailment as c
        ON pn.nationalGridBmUnit = c.nationalGridBmUnit and pn.day = c.day
        WHERE pn.day >= '2025-01-01' and pn.day <= '2025-12-31'
        GROUP BY pn.day, pn.nationalGridBmUnit
        ORDER BY pn.day);

        SELECT * from daily LIMIT 10
        """).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬───────────────────┐
│    day     │ nationalGridBmUnit │   expected_mwh    │     actual_mwh     │   curtailed_mwh    │  curtailed_cost   │
│    date    │      varchar       │      double       │       double       │       double       │      double       │
├────────────┼────────────────────┼───────────────────┼────────────────────┼────────────────────┼───────────────────┤
│ 2025-01-01 │ CMSTW-1            │             745.5 │              745.5 │                0.0 │               0.0 │
│ 2025-01-01 │ WLNYW-1            │            2712.5 │             2712.5 │                0.0 │               0.0 │
│ 2025-01-01 │ SGRWO-2            │            1800.0 │ 276.01666666666665 │ 1523.9833333333333 │        5971.11625 │
│ 2025-01-01 │ DDGNO-4            │            2030.5 │             2030.5 │                0.0 │               0.0 │
│ 2025-01-01 │ GRGBW-2            │          3099.275 │ 

This gives a very useful table of expected/actual energy generation per BMU per day. We can now combine this with location data from Power Station Dictionary, format values, and export to a .csv:

In [10]:
con.sql("""
        CREATE OR REPLACE VIEW bmu_dict AS (SELECT TRIM(UNNEST(STRING_SPLIT(ngc_bmu_id, ','))) AS single_bmu_id,
             *
               FROM read_csv('data/ids.csv', ignore_errors=true));
        
        SELECT
                day as date,
                d.nationalGridBmUnit as "Unit ID",
                bmu.bmUnitName as "Unit Name",
                expected_mwh/1000 as "Expected Wind Generation (GWh)",
                actual_mwh/1000 as "Actual Wind Generation (GWh)",
                curtailed_mwh/1000 "Curtailed Wind Generation (GWh)",
                curtailed_cost/1000000 "Cost of Wind Curtailment (£M)",
                COALESCE(longitude,0) as longitude,
                COALESCE(latitude,0) as latitude
        FROM daily as d
        LEFT JOIN 'data/bmu_data.json' as bmu
        ON bmu.nationalGridBmUnit = d.nationalGridBmUnit
        LEFT JOIN bmu_dict
        ON bmu_dict.single_bmu_id = d.nationalGridBmUnit
        LEFT JOIN 'data/plant-locations.csv' as loc
        ON bmu_dict.dictionary_id = loc.dictionary_id
        WHERE day < '2026-01-01'
        ORDER BY day, d.nationalGridBmUnit
        """).to_csv('data/daily_data.csv')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
df = pd.read_csv('data/daily_data.csv')
df.head()

,date,Unit ID,Unit Name,Expected Wind Generation (GWh),Actual Wind Generation (GWh),Curtailed Wind Generation (GWh),Cost of Wind Curtailment (£M),longitude,latitude
0,2025-01-01,ABRBO-1,ABRBO-1,1.19775,1.197750,0.000000,0.000000,-1.972800,57.222600
1,2025-01-01,ABRTW-1,Auchrobert Wind Farm,0.27900,0.229317,0.049683,0.004419,-3.988675,55.621764
2,2025-01-01,ACHRW-1,AChruach Wind Farm,0.43200,0.432000,0.000000,0.000000,-5.393722,56.342860
3,2025-01-01,AFTOW-1,Afton Wind Farm,0.53700,0.537000,0.000000,0.000000,-4.169100,55.312800
4,2025-01-01,AIRSW-1,Airies,0.46350,0.463500,0.000000,0.000000,-4.713441,54.970896


In [12]:
print(f"Total Expected Wind Generation: {df['Expected Wind Generation (GWh)'].sum():.2f} GWh")
print(f"Total Actual Wind Generation (GWh): {df['Actual Wind Generation (GWh)'].sum():.2f} GWh")
print(f"Total Curtailed Wind Generation (GWh): {df['Curtailed Wind Generation (GWh)'].sum():.2f} GWh")
print(f"Total Cost of Wind Curtailment (£M): £{df['Cost of Wind Curtailment (£M)'].sum():.2f} million")

Total Expected Wind Generation: 81422.83 GWh
Total Actual Wind Generation (GWh): 70529.47 GWh
Total Curtailed Wind Generation (GWh): 10893.37 GWh
Total Cost of Wind Curtailment (£M): £374.89 million


In [13]:
import plotly.express as px

df['date'] = pd.to_datetime(df['date'])
monthly = df.groupby(df['date'].dt.month)['Cost of Wind Curtailment (£M)'].sum().reset_index()
monthly['month_names'] = pd.to_datetime(monthly['date']).dt.month_name()
monthly = monthly.rename(columns={'date':'month'})

fig = px.bar(monthly, x='month', y='Cost of Wind Curtailment (£M)')
fig.update_xaxes(
    tickvals=list(range(1,13)),
    ticktext=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
)
fig.show()

In [14]:
daily = df.groupby(df['date'].dt.date)[['Expected Wind Generation (GWh)','Actual Wind Generation (GWh)']].sum().reset_index()

fig = px.line(daily, x='date', y=['Expected Wind Generation (GWh)','Actual Wind Generation (GWh)'],
              title='Daily Wind Curtailment')
fig.update_layout(legend=dict(title_text=''))
fig.show()

# Conclusions

- 10.9 TWh of wind generation was curtailed out of a potential 81.4 TWh, roughly a 13% reduction
- This cost the UK £375 million in direct payments for curtailing
- Curtailment occurred the most in June, September, and October